# Task 2: single-head row self-attention

See **Task 2** on the task sheet for the specification. This is the reference solution.

In [ ]:
import math
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import study_checks
except ImportError:  # running inside the repository
    for rel in ("../../../tools/expert_study", "../../../../tools/expert_study"):
        path = os.path.abspath(rel)
        if os.path.isdir(path):
            sys.path.append(path)
    import study_checks

study_checks.start_session("T2")
x, y = study_checks.load_batch()
print("x", tuple(x.shape), x.dtype, "| y", tuple(y.shape), y.dtype)

In [ ]:
# Allowed: nn.Linear, nn.ReLU / torch.relu, nn.Flatten / torch.flatten, @ / torch.matmul,
#   transpose, scalar multiplication, softmax, torch.cat, +, nn.ModuleList, Python loops,
#   view / reshape (to split heads), F.cross_entropy / nn.CrossEntropyLoss.
# Forbidden: nn.MultiheadAttention, F.scaled_dot_product_attention,
#   F.multi_head_attention_forward, nn.Transformer*.

class RowAttentionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(28, 32)
        self.classifier = nn.Linear(28 * 32, 10)
        self.q = nn.Linear(32, 32)
        self.k = nn.Linear(32, 32)
        self.v = nn.Linear(32, 32)

    def forward(self, x):
        h = self.embed(torch.flatten(x, 1, 2))
        scores = self.q(h) @ self.k(h).transpose(-2, -1) * (1 / math.sqrt(32))
        attn = torch.softmax(scores, dim=-1) @ self.v(h)
        h = h + attn
        return self.classifier(torch.flatten(h, 1))

In [ ]:
def compute_loss(logits, target):
    return F.cross_entropy(logits, target)  # provided

In [ ]:
study_checks.check_task2(RowAttentionClassifier, compute_loss);